In [1]:
from pathlib import Path
from itertools import product
import hashlib
import json
import warnings
import numpy as np
import pandas as pd
import statsmodels.api as sm
import yfinance as yf
from arch import arch_model
from scipy.stats import kstest
from statsmodels.stats.diagnostic import acorr_ljungbox, het_arch

OUTPUT_DIRECTORY = Path("Var_model_filter_sensitivity")
TAIL_PROBABILITIES = [0.01, 0.03, 0.05]
FORECAST_HORIZONS = [1, 5, 10, 21]
DISTRIBUTIONS = ["normal", "t"]
ESTIMATION_WINDOW = 1260
START_DATE = "2007-10-01"
END_DATE = "2025-10-30"
DIAGNOSTIC_LAGS = 10
CHECKPOINT_INTERVAL = 100
HORIZON_LABEL = "Forecast horizon (trading days)"
MODEL_NAMES = {
    "normal": "Gaussian GARCH(2,2)–FHS",
    "t": "Student-t GARCH(2,2)–FHS",
}
SAMPLES = {
    "Full sample": (START_DATE, END_DATE),
    "Global Financial Crisis": ("2008-01-01", "2009-12-31"),
    "COVID-19": ("2020-01-01", "2020-12-31"),
    "Inflation": ("2022-01-01", "2022-12-31"),
}
HAC_LAGS = {
    "Full sample": 10,
    "Global Financial Crisis": 5,
    "COVID-19": 5,
    "Inflation": 5,
}


def load_returns():
    path = OUTPUT_DIRECTORY / "SP500_returns.csv"

    if path.exists():
        returns = pd.read_csv(
            path, index_col=0, parse_dates=True
        ).iloc[:, 0]
    else:
        prices = yf.download(
            "^GSPC",
            start="2000-01-01",
            end=(pd.Timestamp(END_DATE) + pd.Timedelta(days=1)).strftime("%Y-%m-%d"),
            auto_adjust=True,
            progress=False,
        )
        close = prices["Close"]

        if isinstance(close, pd.DataFrame):
            close = close.iloc[:, 0]

        close = close.dropna().sort_index()

        if close.empty or (close <= 0).any():
            raise ValueError("Missing or invalid closing prices.")

        returns = np.log(close).diff().dropna().rename("Return")

    returns = returns.astype(float).dropna().sort_index()
    returns.index = pd.DatetimeIndex(
        returns.index
    ).tz_localize(None).normalize()
    returns = returns.loc[:END_DATE].rename("Return")

    if returns.empty:
        raise ValueError("No return observations.")

    if returns.index.has_duplicates or not np.isfinite(returns).all():
        raise ValueError("Duplicate dates or nonfinite returns.")

    returns.to_csv(path)
    return returns


def fit_filter(window, distribution):
    model = arch_model(
        100 * window,
        mean="Constant",
        vol="GARCH",
        p=2,
        o=0,
        q=2,
        dist=distribution,
        rescale=False,
    )
    result = model.fit(
        disp="off",
        show_warning=False,
        options={"maxiter": 1000},
    )

    if result.convergence_flag != 0:
        raise ValueError("Optimizer failed to converge.")

    if not np.isfinite(result.params).all():
        raise ValueError("Nonfinite model parameters.")

    return result


def forecast_distributions(result):
    raw_residuals = np.asarray(result.std_resid, dtype=float)
    raw_residuals = raw_residuals[np.isfinite(raw_residuals)]

    if len(raw_residuals) < 2:
        raise ValueError("Insufficient valid residuals.")

    residual_sd = raw_residuals.std(ddof=0)

    if not np.isfinite(residual_sd) or residual_sd <= 0:
        raise ValueError("Invalid residual standard deviation.")

    residuals = (
        raw_residuals - raw_residuals.mean()
    ) / residual_sd

    quantiles = np.quantile(
        residuals,
        TAIL_PROBABILITIES,
        method="inverted_cdf",
    )
    prediction = result.forecast(
        horizon=max(FORECAST_HORIZONS),
        method="analytic",
        reindex=False,
    )
    distributions = {}

    for horizon in FORECAST_HORIZONS:
        mean = float(prediction.mean.iloc[-1, horizon - 1]) / 100
        variance = float(
            prediction.variance.iloc[-1, horizon - 1]
        ) / 10000

        if (
            not np.isfinite(mean)
            or not np.isfinite(variance)
            or variance <= 0
        ):
            raise ValueError(
                f"Invalid daily-return forecast at horizon {horizon}."
            )

        standard_deviation = np.sqrt(variance)
        distributions[horizon] = (
            mean + standard_deviation * quantiles,
            mean,
            standard_deviation,
        )

    return distributions, raw_residuals


def calculate_diagnostics(result, residuals, model_name, origin):
    names = result.model.distribution.parameter_names()
    parameters = np.asarray([result.params[name] for name in names])
    transforms = result.model.distribution.cdf(
        residuals,
        parameters if len(parameters) else None,
    )

    return {
        "Model": model_name,
        "Forecast origin": origin,
        "AIC": float(result.aic),
        "BIC": float(result.bic),
        "Residual LB p-value": float(
            acorr_ljungbox(
                residuals,
                lags=[DIAGNOSTIC_LAGS],
                return_df=True,
            )["lb_pvalue"].iloc[0]
        ),
        "Squared-residual LB p-value": float(
            acorr_ljungbox(
                residuals ** 2,
                lags=[DIAGNOSTIC_LAGS],
                return_df=True,
            )["lb_pvalue"].iloc[0]
        ),
        "ARCH p-value": float(
            het_arch(residuals, nlags=DIAGNOSTIC_LAGS)[1]
        ),
        "PIT KS discrepancy": float(
            kstest(transforms, "uniform").statistic
        ),
        "Residual mean": float(residuals.mean()),
        "Residual standard deviation": float(
            residuals.std(ddof=0)
        ),
        "Diagnostics status": "Estimated",
    }


def run_filter(distribution, returns):
    settings = {
        "implementation": "daily_endpoint_fhs_v2",
        "distribution": distribution,
        "window": ESTIMATION_WINDOW,
        "start": START_DATE,
        "end": END_DATE,
        "tail_probabilities": TAIL_PROBABILITIES,
        "horizons": FORECAST_HORIZONS,
        "diagnostic_lags": DIAGNOSTIC_LAGS,
        "returns_hash": hashlib.sha256(
            returns.to_csv().encode()
        ).hexdigest(),
    }
    signature = hashlib.sha256(
        json.dumps(settings, sort_keys=True).encode()
    ).hexdigest()[:16]
    cache_path = OUTPUT_DIRECTORY / (
        f"filter_{distribution}_{signature}.pkl"
    )

    if cache_path.exists():
        saved = pd.read_pickle(cache_path)
        forecasts = saved["forecasts"]
        diagnostics = saved["diagnostics"]
        completed = set(saved["completed"])
    else:
        forecasts, diagnostics, completed = [], [], set()

    values = returns.to_numpy(dtype=float)
    dates = returns.index
    model_name = MODEL_NAMES[distribution]
    start = pd.Timestamp(START_DATE)
    end = pd.Timestamp(END_DATE)

    def save_progress():
        temporary_path = cache_path.with_suffix(".tmp")
        pd.to_pickle(
            {
                "forecasts": forecasts,
                "diagnostics": diagnostics,
                "completed": sorted(completed),
            },
            temporary_path,
        )
        temporary_path.replace(cache_path)

    for position in range(
        ESTIMATION_WINDOW - 1, len(returns) - 1
    ):
        if position in completed:
            continue

        eligible_horizons = [
            horizon
            for horizon in FORECAST_HORIZONS
            if position + horizon < len(returns)
            and start <= dates[position + horizon] <= end
        ]

        if not eligible_horizons:
            continue

        origin = dates[position]
        window = returns.iloc[
            position - ESTIMATION_WINDOW + 1:position + 1
        ]
        distributions = None
        error = ""

        try:
            result = fit_filter(window, distribution)
            distributions, residuals = forecast_distributions(result)

            try:
                diagnostic = calculate_diagnostics(
                    result, residuals, model_name, origin
                )
            except Exception as exc:
                diagnostic = {
                    "Model": model_name,
                    "Forecast origin": origin,
                    "AIC": float(result.aic),
                    "BIC": float(result.bic),
                    "Residual LB p-value": np.nan,
                    "Squared-residual LB p-value": np.nan,
                    "ARCH p-value": np.nan,
                    "PIT KS discrepancy": np.nan,
                    "Residual mean": float(residuals.mean()),
                    "Residual standard deviation": float(
                        residuals.std(ddof=0)
                    ),
                    "Diagnostics status": (
                        f"{type(exc).__name__}: {exc}"
                    ),
                }

            diagnostics.append(diagnostic)

        except Exception as exc:
            error = f"{type(exc).__name__}: {exc}"

        for horizon in eligible_horizons:
            evaluation = dates[position + horizon]
            actual_return = float(values[position + horizon])

            for index, alpha in enumerate(TAIL_PROBABILITIES):
                model_var = cantelli_var = ratio = np.nan
                mean = standard_deviation = np.nan
                status = error or "Valid"

                if distributions is not None:
                    quantiles, mean, standard_deviation = (
                        distributions[horizon]
                    )
                    model_var = -float(quantiles[index])
                    cantelli_var = (
                        -mean
                        + standard_deviation
                        * np.sqrt((1 - alpha) / alpha)
                    )

                    if (
                        not np.isfinite(cantelli_var)
                        or cantelli_var <= 0
                    ):
                        status = (
                            "Nonpositive or invalid Cantelli benchmark"
                        )
                    elif (
                        not np.isfinite(model_var)
                        or model_var < 0
                    ):
                        status = "Negative or invalid model loss VaR"
                    else:
                        ratio = model_var / cantelli_var

                        if ratio > 1 + 1e-10:
                            status = "Cantelli ordering violation"
                            ratio = np.nan

                forecasts.append({
                    "Model": model_name,
                    "Forecast origin": origin,
                    "Evaluation date": evaluation,
                    "Estimation window start": window.index[0],
                    "Estimation window observations": len(window),
                    "Tail probability": alpha,
                    HORIZON_LABEL: horizon,
                    "Forecast daily return mean": mean,
                    "Forecast daily return standard deviation": (
                        standard_deviation
                    ),
                    "Realised daily log return": actual_return,
                    "Model loss VaR": model_var,
                    "Cantelli loss benchmark": cantelli_var,
                    "VER": ratio,
                    "Breach": (
                        int(-actual_return > model_var)
                        if np.isfinite(model_var) else np.nan
                    ),
                    "Status": status,
                })

        completed.add(position)

        if len(completed) % CHECKPOINT_INTERVAL == 0:
            save_progress()
            print(
                f"{model_name}: {len(completed)} origins completed; "
                f"latest {origin.date()}",
                flush=True,
            )

    save_progress()
    return pd.DataFrame(forecasts), pd.DataFrame(diagnostics)


def match_forecasts(forecasts):
    keys = [
        "Forecast origin",
        "Evaluation date",
        "Tail probability",
        HORIZON_LABEL,
    ]
    valid = forecasts.loc[
        forecasts["Status"].eq("Valid")
        & np.isfinite(forecasts["VER"])
    ].copy()

    counts = valid.groupby(keys)["Model"].nunique()
    shared = counts.loc[
        counts.eq(len(DISTRIBUTIONS))
    ].reset_index()[keys]

    return valid.merge(
        shared,
        on=keys,
        how="inner",
        validate="many_to_one",
    )


def estimate_mean_difference(data, lags):
    if data["Breach"].nunique() < 2:
        return np.nan, np.nan, np.nan, "Missing outcome group"

    try:
        design = sm.add_constant(
            data[["Breach"]].astype(float),
            has_constant="add",
        )

        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            result = sm.OLS(
                data["VER"].astype(float),
                design,
            ).fit(
                cov_type="HAC",
                cov_kwds={
                    "maxlags": int(lags),
                    "kernel": "bartlett",
                    "use_correction": False,
                },
            )

        interval = result.conf_int().loc["Breach"]
        estimates = [
            result.pvalues["Breach"],
            interval.iloc[0],
            interval.iloc[1],
        ]

        if not np.isfinite(estimates).all():
            return np.nan, np.nan, np.nan, "Nonfinite inference"

        return (
            float(estimates[0]),
            float(estimates[1]),
            float(estimates[2]),
            "Estimated",
        )

    except Exception as exc:
        return (
            np.nan,
            np.nan,
            np.nan,
            f"{type(exc).__name__}: {exc}",
        )


def define_states(values):
    if len(values) == 0:
        return {
            "Quartiles": (
                np.array([], dtype=int),
                ["Q1", "Q2", "Q3", "Q4"],
            ),
            "Three states": (
                np.array([], dtype=int),
                ["Low", "Normal", "High"],
            ),
        }

    first, second, third = np.quantile(
        values, [0.25, 0.50, 0.75]
    )
    lower, upper = np.quantile(values, [0.33, 0.67])

    quartiles = np.select(
        [values <= first, values <= second, values <= third],
        [0, 1, 2],
        default=3,
    )
    three_states = np.where(
        values <= lower,
        0,
        np.where(values >= upper, 2, 1),
    )

    return {
        "Quartiles": (
            quartiles,
            ["Q1", "Q2", "Q3", "Q4"],
        ),
        "Three states": (
            three_states,
            ["Low", "Normal", "High"],
        ),
    }


def diagnostic_summary(diagnostics):
    rows = []
    counts = diagnostics.groupby(
        "Forecast origin"
    )["Model"].nunique()
    matched_origins = counts.index[
        counts.eq(len(DISTRIBUTIONS))
    ]

    for model_name in MODEL_NAMES.values():
        for sample, (start, end) in SAMPLES.items():
            data = diagnostics.loc[
                diagnostics["Model"].eq(model_name)
                & diagnostics["Forecast origin"].isin(
                    matched_origins
                )
                & diagnostics["Forecast origin"].between(
                    start, end
                )
            ]

            row = {
                "Model": model_name,
                "Sample": sample,
                "Matched diagnostic windows": len(data),
                "Median AIC": data["AIC"].median(),
                "Median BIC": data["BIC"].median(),
                "Median PIT KS discrepancy": data[
                    "PIT KS discrepancy"
                ].median(),
                "Diagnostic lag count": DIAGNOSTIC_LAGS,
            }

            for column, label in [
                ("Residual LB p-value", "Residual LB"),
                (
                    "Squared-residual LB p-value",
                    "Squared-residual LB",
                ),
                ("ARCH p-value", "ARCH"),
            ]:
                available = data[column].dropna()
                row[f"{label} valid windows"] = len(available)
                row[f"{label} rejection share (5%)"] = (
                    float(available.lt(0.05).mean())
                    if len(available) else np.nan
                )

            rows.append(row)

    return pd.DataFrame(rows)


def compare_results(forecasts, matched, returns):
    differences, quartiles, transitions = [], [], []

    for model_name in MODEL_NAMES.values():
        for sample, (start, end) in SAMPLES.items():
            for alpha, horizon in product(
                TAIL_PROBABILITIES, FORECAST_HORIZONS
            ):
                raw = forecasts.loc[
                    forecasts["Model"].eq(model_name)
                    & forecasts["Evaluation date"].between(
                        start, end
                    )
                    & forecasts["Tail probability"].eq(alpha)
                    & forecasts[HORIZON_LABEL].eq(horizon)
                ]
                data = matched.loc[
                    matched["Model"].eq(model_name)
                    & matched["Evaluation date"].between(
                        start, end
                    )
                    & matched["Tail probability"].eq(alpha)
                    & matched[HORIZON_LABEL].eq(horizon)
                ].sort_values("Forecast origin")

                base = {
                    "Model": model_name,
                    "Sample": sample,
                    "Tail probability": alpha,
                    HORIZON_LABEL: horizon,
                }

                breach = data.loc[
                    data["Breach"].eq(1), "VER"
                ]
                nonbreach = data.loc[
                    data["Breach"].eq(0), "VER"
                ]
                lags = HAC_LAGS[sample]

                p_value, lower, upper, status = (
                    estimate_mean_difference(data, lags)
                )

                differences.append({
                    **base,
                    "Observations": len(data),
                    "Excluded or unmatched forecasts": (
                        len(raw) - len(data)
                    ),
                    "Breaches": len(breach),
                    "Mean VER on breach observations": (
                        breach.mean()
                    ),
                    "Mean VER on non-breach observations": (
                        nonbreach.mean()
                    ),
                    "Mean difference": (
                        breach.mean() - nonbreach.mean()
                    ),
                    "HAC p-value": p_value,
                    "95% confidence interval lower": lower,
                    "95% confidence interval upper": upper,
                    "HAC lags": lags,
                    "Status": status,
                })

                definitions = define_states(
                    data["VER"].to_numpy()
                )
                labels = definitions["Quartiles"][0]
                rates = []

                quartile_row = {
                    **base,
                    "Observations": len(data),
                    "Breaches": len(breach),
                }

                for quartile in range(4):
                    group = data.iloc[
                        np.flatnonzero(labels == quartile)
                    ]
                    frequency = group["Breach"].mean()
                    rates.append(frequency)

                    quartile_row.update({
                        f"Q{quartile + 1} observations": len(group),
                        f"Q{quartile + 1} breaches": int(
                            group["Breach"].sum()
                        ),
                        f"Q{quartile + 1} breach frequency": (
                            frequency
                        ),
                    })

                informative = (
                    data["Breach"].nunique() == 2
                    and np.isfinite(rates).all()
                )

                quartile_row.update({
                    "Nonincreasing Q1–Q4 ordering": (
                        bool(np.all(np.diff(rates) <= 0))
                        if informative else np.nan
                    ),
                    "Q1 frequency exceeds Q4": (
                        bool(rates[0] > rates[3])
                        if informative else np.nan
                    ),
                    "Status": (
                        "Descriptive"
                        if informative
                        else "Insufficient outcome or group variation"
                    ),
                })
                quartiles.append(quartile_row)

                positions = returns.index.get_indexer(
                    data["Forecast origin"]
                )
                adjacent = np.diff(positions) == 1

                for definition, (states, names) in definitions.items():
                    destinations = states[1:][adjacent]

                    for source, target in product(
                        range(len(names)), repeat=2
                    ):
                        eligible = (
                            (states[:-1] == source) & adjacent
                        )
                        count = int(
                            (
                                eligible
                                & (states[1:] == target)
                            ).sum()
                        )
                        total = int(eligible.sum())
                        probability = (
                            count / total if total else np.nan
                        )
                        source_share = (
                            float(
                                (destinations == source).mean()
                            )
                            if len(destinations) else np.nan
                        )

                        persistence = np.nan

                        if (
                            source == target
                            and np.isfinite(probability)
                            and np.isfinite(source_share)
                            and source_share > 0
                        ):
                            persistence = (
                                probability / source_share
                            )

                        transitions.append({
                            **base,
                            "State definition": definition,
                            "From state": names[source],
                            "To state": names[target],
                            "Transitions": count,
                            "Transitions from source state": total,
                            "Transition probability": probability,
                            "Destination sample share of source state": (
                                source_share
                            ),
                            "Persistence relative to sample share": (
                                persistence
                            ),
                            "Status": (
                                "Descriptive"
                                if total
                                else "No transitions from source state"
                            ),
                        })

    return (
        pd.DataFrame(differences),
        pd.DataFrame(quartiles),
        pd.DataFrame(transitions),
    )


def format_p_value(value):
    if pd.isna(value):
        return "N/A"
    return "< 0.001" if value < 0.001 else f"{value:.3f}"


def export_table(table, path):
    report = table.copy()

    for column in report.columns:
        if "p-value" in column:
            report[column] = report[column].map(
                format_p_value
            )

    report.to_csv(
        path,
        index=False,
        na_rep="N/A",
    )


def main():
    OUTPUT_DIRECTORY.mkdir(
        parents=True, exist_ok=True
    )
    returns = load_returns()

    results = [
        run_filter(distribution, returns)
        for distribution in DISTRIBUTIONS
    ]

    forecasts = pd.concat(
        [result[0] for result in results],
        ignore_index=True,
    )
    diagnostics = pd.concat(
        [result[1] for result in results],
        ignore_index=True,
    )

    if forecasts.empty or diagnostics.empty:
        raise RuntimeError(
            "No usable forecasts or diagnostics."
        )

    matched = match_forecasts(forecasts)

    if matched.empty:
        raise RuntimeError(
            "No valid observations shared by both filters."
        )

    differences, quartiles, transitions = compare_results(
        forecasts, matched, returns
    )

    outputs = {
        "Filter_diagnostics_summary.csv": (
            diagnostic_summary(diagnostics)
        ),
        "VER_breach_comparison.csv": differences,
        "VER_quartile_ordering.csv": quartiles,
        "VER_Markov_transitions.csv": transitions,
        "Filter_diagnostics_by_origin.csv": diagnostics,
    }

    for filename, table in outputs.items():
        export_table(
            table, OUTPUT_DIRECTORY / filename
        )

    forecasts.to_csv(
        OUTPUT_DIRECTORY / "Daily_return_forecasts.csv",
        index=False,
        na_rep="N/A",
    )

   
    print(
        f"Results saved to {OUTPUT_DIRECTORY.resolve()}"
    )


if __name__ == "__main__":
    main()

Gaussian GARCH(2,2)–FHS: 100 origins completed; latest 2008-01-23
Gaussian GARCH(2,2)–FHS: 200 origins completed; latest 2008-06-16
Gaussian GARCH(2,2)–FHS: 300 origins completed; latest 2008-11-05
Gaussian GARCH(2,2)–FHS: 400 origins completed; latest 2009-04-01
Gaussian GARCH(2,2)–FHS: 500 origins completed; latest 2009-08-24
Gaussian GARCH(2,2)–FHS: 600 origins completed; latest 2010-01-15
Gaussian GARCH(2,2)–FHS: 700 origins completed; latest 2010-06-10
Gaussian GARCH(2,2)–FHS: 800 origins completed; latest 2010-11-01
Gaussian GARCH(2,2)–FHS: 900 origins completed; latest 2011-03-25
Gaussian GARCH(2,2)–FHS: 1000 origins completed; latest 2011-08-17
Gaussian GARCH(2,2)–FHS: 1100 origins completed; latest 2012-01-10
Gaussian GARCH(2,2)–FHS: 1200 origins completed; latest 2012-06-04
Gaussian GARCH(2,2)–FHS: 1300 origins completed; latest 2012-10-24
Gaussian GARCH(2,2)–FHS: 1400 origins completed; latest 2013-03-22
Gaussian GARCH(2,2)–FHS: 1500 origins completed; latest 2013-08-14
Gaus